# n3_train_image_conditional_diffusion.ipynb

This notebook trains an **image-conditioned diffusion model** for pixel-art character animation in the PIXEL-T2I project.

The model is trained on **50,000 RGBA pixel-art characters**, where each sample consists of:
- a **4-view character sprite** (128x128, front/back/left/right), and  
- a corresponding **action animation**, represented as individual **64x64 action tiles** (walk, thrust, slash).

Instead of generating the full action sprite sheet directly, the model is trained at the **tile level**, predicting **one action frame at a time** conditioned on:
- the input 4-view character image, and  
- an explicit **frame identifier** encoding action type, direction, and temporal index.

During inference, the generated tiles are **assembled back into a full 576x768 action sprite sheet**, preserving the original LPC-style layout.

This image-conditional diffusion model extends the unconditional baseline (*n1*) by introducing structured visual conditioning and frame-aware control, enabling **consistent multi-frame action generation** from a single character image.

Training takes approximately **6 hours on an NVIDIA A100 GPU**, and model checkpoints and validation samples are saved for later qualitative evaluation and comparison.

## Section 1. Setup and Imports

In [ ]:
# Install required packages
!pip install -q accelerate

# Standard library
import os
import math
import time
import warnings
from pathlib import Path
from typing import List, Tuple, Dict
import csv
from datetime import datetime

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils as nn_utils
from torch.utils.data import Dataset, DataLoader, Subset
from torch.cuda.amp import autocast, GradScaler

# PyTorch optimization
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

# Vision / Image processing
from torchvision.utils import save_image, make_grid
from PIL import Image
import torchvision.transforms as T

# Visualization
import matplotlib.pyplot as plt

# Progress bar
from tqdm.auto import tqdm

print("All libraries imported successfully")

## Section 2. Environment & Device Setup

In [ ]:
# 2.1 Environment check with memory optimization
import os
import torch

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128,expandable_segments:True'

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    try:
        torch.cuda.set_per_process_memory_fraction(0.85)
    except Exception as e:
        print(f"WARNING: set_per_process_memory_fraction failed: {e}")
    device = torch.device("cuda")

    props = torch.cuda.get_device_properties(0)
    print(f"GPU memory: {props.total_memory / 1024**3:.1f} GB")
else:
    device = torch.device("cpu")
    print("WARNING: No GPU available!")

# 2.2 Mount Google Drive and prepare datasets
from google.colab import drive
import shutil
import zipfile

drive.mount('/content/drive')

# Drive paths
DRIVE_ROOT = Path("/content/drive/MyDrive/PIXEL-T2I").resolve()

# Dataset zip paths
DRIVE_4VIEW_ZIP = DRIVE_ROOT / "pixel_character_dataset" / "processed" / "dataset_4view.zip"
DRIVE_ACTIONS_ZIP = DRIVE_ROOT / "pixel_character_dataset" / "processed" / "dataset_actions.zip"

# Local SSD paths (fast reading)
LOCAL_DATA = Path("/content/data_cache")
LOCAL_4VIEW = LOCAL_DATA / "dataset_4view"
LOCAL_ACTIONS = LOCAL_DATA / "dataset_actions"

# 4-view dataset paths
FOURVIEW_DIR = LOCAL_4VIEW / "images"
FOURVIEW_CSV = LOCAL_4VIEW / "captions.csv"

# Actions dataset paths
ACTIONS_DIR = LOCAL_ACTIONS / "images"

# 2.3 Copy 4-view dataset to local SSD
if not FOURVIEW_DIR.exists():
    print(f"Copying 4-view dataset from Drive...")

    LOCAL_DATA.mkdir(parents=True, exist_ok=True)
    local_zip_path = Path("/content/temp_4view.zip")

    shutil.copy(DRIVE_4VIEW_ZIP, local_zip_path)

    with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_DATA)

    os.remove(local_zip_path)
    print("4-view dataset copied to local SSD")
else:
    print("4-view dataset already in local SSD")

# 2.4 Copy actions dataset to local SSD
if not ACTIONS_DIR.exists():
    print(f"Copying actions dataset from Drive...")

    local_zip_path = Path("/content/temp_actions.zip")

    shutil.copy(DRIVE_ACTIONS_ZIP, local_zip_path)

    with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_DATA)

    os.remove(local_zip_path)
    print("Actions dataset copied to local SSD")
else:
    print("Actions dataset already in local SSD")

# 2.5 Verify datasets
if not FOURVIEW_DIR.exists():
    raise FileNotFoundError(f"Cannot find 4-view images at {FOURVIEW_DIR}")
if not ACTIONS_DIR.exists():
    raise FileNotFoundError(f"Cannot find actions images at {ACTIONS_DIR}")

num_4view = len(list(FOURVIEW_DIR.glob("*.png")))
num_actions = len(list(ACTIONS_DIR.glob("*.png")))
print(f"\n4-view dataset ready: {num_4view} images at {FOURVIEW_DIR}")
print(f"Actions dataset ready: {num_actions} images at {ACTIONS_DIR}")

# Verify two datasets have same count (paired data)
if num_4view != num_actions:
    raise ValueError(f"Dataset mismatch: 4-view ({num_4view}) != actions ({num_actions})")
else:
    print(f"Datasets are paired: {num_4view} image pairs")

# 2.6 Set paths
ROOT = DRIVE_ROOT
DATASET_4VIEW_ROOT = LOCAL_4VIEW
DATASET_ACTIONS_ROOT = LOCAL_ACTIONS

print("\nSection 2 complete")

## Section 3. Configuration (image-conditional)

In [ ]:
CONFIG = {
    # Data paths
    "fourview_dir": FOURVIEW_DIR,          # Directory containing 2x2 four-view character sprites
    "actions_dir": ACTIONS_DIR,            # Directory containing combined action sheet sprites

    # Model and output paths
    "checkpoint_dir": ROOT / "models" / "pixel_image_conditional" / "checkpoints",  # Training checkpoints
    "final_model_dir": ROOT / "models" / "pixel_image_conditional",                  # Final trained model storage
    "log_dir": ROOT / "outputs" / "logs_image_conditional",                          # Training logs
    "sample_dir": ROOT / "outputs" / "samples_image_conditional",                   # Validation samples

    # Image settings
    "fourview_size": 128,                  # Resolution of input 4-view images (square)
    "tile_size": 64,                       # Each action frame is 64x64 (no resizing)
    "sheet_cols": 9,                       # Output sheet always uses 9 columns
    "sheet_rows": 12,                      # 3 actions × 4 directions = 12 rows

    # Actions layout
    "actions": ["walk", "thrust", "slash"],  # Supported action types
    "action_cols": {                       # Number of valid frames per action
        "walk": 9,
        "thrust": 8,
        "slash": 6,
    },

    # Direction order
    "dirs": ["west", "east", "south", "north"],  # Clockwise LPC 4-view order

    # Model channels
    "in_channels": 4,                      # RGBA input
    "out_channels": 4,                     # RGBA output

    # DataLoader settings
    "batch_size": 64,                      # Training batch size
    "num_workers": 2,                      # Data loading workers
    "shuffle": True,                       # Shuffle dataset each epoch
    "pin_memory": True,                    # Pin memory for faster GPU transfer

    # Diffusion parameters
    "timesteps": 1000,                     # Number of diffusion steps
    "beta_schedule": "linear",             # Noise schedule type
    "beta_start": 0.0001,                  # Starting beta
    "beta_end": 0.02,                      # Ending beta

    # Image conditioning
    "num_frames": 108,                     # Total action frame indices
    "char_encoder_channels": [64, 128, 256, 512],  # Character encoder channels
    "char_feature_dim": 512,               # Character feature dimension
    "char_spatial_size": 16,               # Spatial size of character feature map
    "cfg_drop_prob": 0.1,                  # Conditioning dropout probability
    "cfg_scale": 1.0,                      # Classifier-free guidance scale

    # UNet architecture
    "unet_channels": 128,                  # Base UNet channel width
    "channel_mult": (1, 2, 3, 4),           # Channel multipliers per resolution
    "num_res_blocks": 2,                   # Residual blocks per stage
    "attention_resolutions": (16, 8),       # Resolutions using attention
    "dropout": 0.1,                        # UNet dropout rate
    "num_heads": 8,                        # Attention heads
    "use_cross_attention": True,            # Enable cross-attention

    # Training settings
    "num_epochs": 10,                      # Total training epochs
    "learning_rate": 1e-4,                 # Initial learning rate
    "weight_decay": 0.01,                  # Optimizer weight decay
    "lr_scheduler": "cosine",               # Learning rate scheduler
    "warmup_steps": 50,                    # Scheduler warm-up steps
    "gradient_clip": 1.0,                  # Gradient clipping threshold
    "use_mixed_precision": True,            # Enable AMP training

    # Training control
    "start_training": True,                # Start training when script runs
    "resume_training": False,               # Resume from latest checkpoint

    # Logging and checkpointing
    "log_every": 50,                       # Log metrics every N steps
    "save_every_epochs": 1,                # Save checkpoint every N epochs
    "keep_last_n": 3,                      # Keep last N checkpoints
    "sample_every_epochs": 1,              # Sample validation images every N epochs
    "num_samples": 4,                      # Samples per validation run

    # Validation sampling
    "save_full_sheet": False,              # Enable full-sheet validation sampling
    "full_sheet_every_epochs": 5,           # Interval for full-sheet sampling

    # Sampling settings
    "default_ddim_steps": 50,               # Default DDIM steps for inference

    # Testing and debugging
    "quick_test": False,                   # Enable fast debug mode
    "quick_test_samples": 8,               # Samples used in debug mode

    # Epoch sampling control
    "max_tiles_per_epoch": 200_000,         # Maximum tile samples per epoch (None to disable)
}

print("Configuration loaded successfully")
print(f"\nKey settings:")
print(f"  4-view images:  {CONFIG['fourview_dir']}")
print(f"  Actions images: {CONFIG['actions_dir']}")
print(f"  4-view size:    {CONFIG['fourview_size']}x{CONFIG['fourview_size']}")
print(f"  Tile size:      {CONFIG['tile_size']}x{CONFIG['tile_size']}")
print(f"  Sheet layout:   {CONFIG['sheet_rows']} rows x {CONFIG['sheet_cols']} cols")
print(f"  Action cols:    {CONFIG['action_cols']}")
print(f"  Batch size:     {CONFIG['batch_size']}")
print(f"  Epochs:         {CONFIG['num_epochs']}")
print(f"  Quick test:     {CONFIG['quick_test']}")

print("\nSection 3 complete")

## Section 4. Image Encoder

In [ ]:
class SpriteCharacterEncoder(nn.Module):
    """
    Character encoder for 4-view sprite images.
    Encodes character appearance into spatial features for conditioning.

    Input:  [B, 4, 128, 128] - RGBA 4-view sprites
    Output: [B, 512, 16, 16] - Spatial character features
    """
    def __init__(
        self,
        in_channels=4,
        base_channels=64,
        channel_progression=[64, 128, 256, 512],
        output_dim=512,
        spatial_size=16
    ):
        super().__init__()

        self.in_channels = in_channels
        self.output_dim = output_dim
        self.spatial_size = spatial_size

        # Build encoder blocks
        self.blocks = nn.ModuleList()

        # Initial conv
        prev_ch = in_channels

        for i, out_ch in enumerate(channel_progression):
            # Determine stride: downsample for first 3 blocks, keep spatial for last
            # 128 -> 64 -> 32 -> 16 -> 16
            stride = 2 if i < 3 else 1

            block = self._make_encoder_block(
                in_ch=prev_ch,
                out_ch=out_ch,
                stride=stride
            )
            self.blocks.append(block)
            prev_ch = out_ch

        # Final projection to desired output dim (if needed)
        if channel_progression[-1] != output_dim:
            self.final_proj = nn.Conv2d(channel_progression[-1], output_dim, 1)
        else:
            self.final_proj = nn.Identity()

    def _make_encoder_block(self, in_ch, out_ch, stride):
        """Create a residual encoder block"""
        return nn.Sequential(
            # First conv (with downsampling if stride=2)
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1),
            nn.GroupNorm(num_groups=min(32, max(1, out_ch // 4)), num_channels=out_ch),
            nn.SiLU(),

            # Second conv (no downsampling)
            nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1),
            nn.GroupNorm(num_groups=min(32, max(1, out_ch // 4)), num_channels=out_ch),
            nn.SiLU(),
        )

    def forward(self, x):
        """
        Args:
            x: [B, 4, 128, 128] - 4-view character sprites

        Returns:
            features: [B, output_dim, spatial_size, spatial_size] - Character features
        """
        # Encode through blocks
        for block in self.blocks:
            x = block(x)

        # Final projection
        x = self.final_proj(x)

        return x


# Test the encoder
def test_character_encoder():
    """Test character encoder with sample input"""
    print("Testing Character Encoder...")

    # Create encoder
    encoder = SpriteCharacterEncoder(
        in_channels=CONFIG['in_channels'],
        base_channels=CONFIG['char_encoder_channels'][0],
        channel_progression=CONFIG['char_encoder_channels'],
        output_dim=CONFIG['char_feature_dim'],
        spatial_size=CONFIG['char_spatial_size']
    ).to(device)

    # Test input
    batch_size = 2
    test_input = torch.randn(
        batch_size,
        CONFIG['in_channels'],
        CONFIG['fourview_size'],
        CONFIG['fourview_size']
    ).to(device)

    # Forward pass
    with torch.no_grad():
        output = encoder(test_input)

    # Verify output shape
    expected_shape = (batch_size, CONFIG['char_feature_dim'], CONFIG['char_spatial_size'], CONFIG['char_spatial_size'])
    assert output.shape == expected_shape, f"Expected {expected_shape}, got {output.shape}"

    # Print model info
    total_params = sum(p.numel() for p in encoder.parameters())
    trainable_params = sum(p.numel() for p in encoder.parameters() if p.requires_grad)

    print(f"Character Encoder Test Passed...")
    print(f"  Input shape:  {test_input.shape}")
    print(f"  Output shape: {output.shape}")
    print(f"  Total params: {total_params:,}")
    print(f"  Trainable:    {trainable_params:,}")
    print(f"  Memory:       ~{total_params * 4 / 1024 / 1024:.2f} MB (fp32)")

    return encoder


# Run test
char_encoder = test_character_encoder()

print("\nSection 4 complete")

## Section 5. Dataset & DataLoader

In [ ]:
# 5.1 Dataset for paired 4-view and actions sprites (Image Conditional)
class PairedSpriteTileDataset(Dataset):
    """
    Image-conditional sprite dataset with paired 4-view and actions images.

    Instead of predicting the whole action sheet, we predict ONE 64x64 tile.

    Structure:
        fourview_dir/character_00000.png  ←→  actions_dir/character_00000.png
        fourview_dir/character_00001.png  ←→  actions_dir/character_00001.png
        ...
    """
    def __init__(self, config):
        self.config = config
        self.fourview_dir = Path(config["fourview_dir"])
        self.actions_dir = Path(config["actions_dir"])
        self.fourview_size = config["fourview_size"]

        # Sheet / tile config
        self.tile_size = config["tile_size"]
        self.sheet_cols = config["sheet_cols"]
        self.sheet_rows = config["sheet_rows"]
        self.actions = config["actions"]
        self.action_cols = config["action_cols"]
        self.dirs = config["dirs"]

        # Build mappings
        self.action_to_id = {a: i for i, a in enumerate(self.actions)}
        self.dir_to_id = {d: i for i, d in enumerate(self.dirs)}

        # Load image filenames (from 4-view directory)
        self.image_files = sorted([f for f in os.listdir(self.fourview_dir) if f.endswith('.png')])

        # Verify paired data exists
        self._verify_paired_data()

        print(f"Loaded {len(self.image_files)} paired images")
        print(f"  4-view dir:  {self.fourview_dir}")
        print(f"  Actions dir: {self.actions_dir}")

        # Define transforms
        self.fourview_transform = self._get_fourview_transforms()

        # Precompute valid (action_id, dir_id, t_id) triples
        self.valid_frames = self._build_valid_frames()
        print(f"Built {len(self.valid_frames)} valid frames per character")

    def _verify_paired_data(self):
        """Verify that all 4-view images have corresponding actions images"""
        missing_actions = []

        for img_file in self.image_files:
            actions_path = self.actions_dir / img_file
            if not actions_path.exists():
                missing_actions.append(img_file)

        if missing_actions:
            print(f"Warning: {len(missing_actions)} files missing in actions directory")
            if len(missing_actions) <= 5:
                print(f"  Missing: {missing_actions}")

            # Remove files without pairs
            self.image_files = [f for f in self.image_files if f not in missing_actions]
            print(f"  Filtered to {len(self.image_files)} valid pairs")

    def _get_fourview_transforms(self):
        """Define transformations for 4-view input images"""
        return T.Compose([
            T.Lambda(lambda img: img.convert('RGBA') if img.mode != 'RGBA' else img),
            T.Resize((self.fourview_size, self.fourview_size), interpolation=T.InterpolationMode.NEAREST),
            T.ToTensor(),
            T.Normalize(mean=[0.5]*4, std=[0.5]*4),  # Normalize to [-1, 1]
        ])

    def _build_valid_frames(self) -> List[Tuple[int, int, int]]:
        """
        Build the list of valid frames (action_id, dir_id, t_id).

        Layout:
          - walk:   4 rows, 9 cols
          - thrust: 4 rows, 8 cols
          - slash:  4 rows, 6 cols
        """
        frames = []
        for action_name in self.actions:
            a_id = self.action_to_id[action_name]
            max_t = self.action_cols[action_name]  # number of valid columns
            for d_id in range(len(self.dirs)):
                for t_id in range(max_t):
                    frames.append((a_id, d_id, t_id))
        return frames

    def _sheet_row_from_ids(self, action_id: int, dir_id: int) -> int:
        """Map (action_id, dir_id) to sheet row index [0..11]"""
        return action_id * 4 + dir_id

    def _frame_id_from_ids(self, action_id: int, dir_id: int, t_id: int) -> int:
        """Flatten (action_id, dir_id, t_id) into a single frame_id [0..107]"""
        return action_id * (4 * self.sheet_cols) + dir_id * self.sheet_cols + t_id

    def __len__(self):
        # Each character provides multiple training samples (one per valid frame)
        return len(self.image_files) * len(self.valid_frames)

    def __getitem__(self, idx):
        """
        Returns:
            dict: {
                'fourview':  tensor (4, 128, 128),
                'tile':      tensor (4, 64, 64),
                'action_id': int,
                'dir_id':    int,
                't_id':      int,
                'frame_id':  int,
                'filename':  str
            }
        """
        # Decode global index -> (character index, frame index)
        frames_per_char = len(self.valid_frames)
        char_idx = idx // frames_per_char
        frame_idx = idx % frames_per_char

        img_name = self.image_files[char_idx]
        fourview_path = self.fourview_dir / img_name
        actions_path = self.actions_dir / img_name

        action_id, dir_id, t_id = self.valid_frames[frame_idx]
        frame_id = self._frame_id_from_ids(action_id, dir_id, t_id)

        # Load 4-view image
        try:
            fourview = Image.open(fourview_path)
            fourview = self.fourview_transform(fourview)
        except Exception as e:
            print(f"Error loading 4-view {fourview_path}: {e}")
            fourview = torch.zeros(4, self.fourview_size, self.fourview_size)

        # Load actions sheet and crop the target tile (64x64)
        try:
            sheet = Image.open(actions_path)
            sheet = sheet.convert('RGBA') if sheet.mode != 'RGBA' else sheet

            # Compute crop box in pixels
            sheet_row = self._sheet_row_from_ids(action_id, dir_id)
            x0 = t_id * self.tile_size
            y0 = sheet_row * self.tile_size
            x1 = x0 + self.tile_size
            y1 = y0 + self.tile_size

            tile = sheet.crop((x0, y0, x1, y1))

            # Convert to tensor and normalize to [-1, 1]
            tile = T.ToTensor()(tile)
            tile = T.Normalize(mean=[0.5]*4, std=[0.5]*4)(tile)

        except Exception as e:
            print(f"Error loading/cropping actions {actions_path}: {e}")
            tile = torch.zeros(4, self.tile_size, self.tile_size)

        return {
            'fourview': fourview,
            'tile': tile,
            'action_id': torch.tensor(action_id, dtype=torch.long),
            'dir_id': torch.tensor(dir_id, dtype=torch.long),
            't_id': torch.tensor(t_id, dtype=torch.long),
            'frame_id': torch.tensor(frame_id, dtype=torch.long),
            'filename': img_name
        }


# 5.2 DataLoader Creation
def create_dataloader(config):
    """Create training dataloader for paired sprite data."""
    dataset = PairedSpriteTileDataset(config)

    # Quick test mode
    if config.get("quick_test", False):
        n_chars = config.get("quick_test_samples", 100)
        n_chars = min(n_chars, len(dataset.image_files))

        # Subset by characters
        frames_per_char = len(dataset.valid_frames)
        subset_len = n_chars * frames_per_char
        dataset = Subset(dataset, range(subset_len))
        print(f"Quick test mode: using {n_chars} characters ({subset_len} samples)")

    else:
        # Limit tiles per epoch
        max_tiles = config.get("max_tiles_per_epoch", None)
        if max_tiles is not None:
            max_tiles = int(max_tiles)
            max_tiles = min(max_tiles, len(dataset))

            # Random subset each run
            perm = torch.randperm(len(dataset))[:max_tiles].tolist()
            dataset = Subset(dataset, perm)

            print(f"Tile sampling: using {max_tiles} / {len(dataset.dataset)} samples this epoch")

    # Create dataloader
    train_loader = DataLoader(
        dataset,
        batch_size=config["batch_size"],
        shuffle=config["shuffle"],
        num_workers=config["num_workers"],
        pin_memory=config["pin_memory"],
        drop_last=True  # Drop last incomplete batch for stable training
    )

    print(f"DataLoader created: {len(dataset)} samples, {len(train_loader)} batches")

    return train_loader


# 5.3 Create DataLoader
train_loader = create_dataloader(CONFIG)


# 5.4 Test DataLoader
print("\nTesting DataLoader...")
test_batch = next(iter(train_loader))
print(f"  4-view batch shape:  {test_batch['fourview'].shape}")
print(f"  Tile batch shape:    {test_batch['tile'].shape}")
print(f"  action_id shape:     {test_batch['action_id'].shape}")
print(f"  dir_id shape:        {test_batch['dir_id'].shape}")
print(f"  t_id shape:          {test_batch['t_id'].shape}")
print(f"  frame_id shape:      {test_batch['frame_id'].shape}")
print(f"  Batch filenames:     {test_batch['filename'][:3]}...")  # Show first 3

# Verify data range
print(f"\n  4-view value range:  [{test_batch['fourview'].min():.3f}, {test_batch['fourview'].max():.3f}]")
print(f"  Tile value range:    [{test_batch['tile'].min():.3f}, {test_batch['tile'].max():.3f}]")

# Expected: [-1, 1] after normalization

print("\nSection 5 complete")

## Section 6. Diffusion Process

In [ ]:
# 6.1 Noise Schedule Functions
def linear_beta_schedule(timesteps, beta_start=0.0001, beta_end=0.02):
    """Linear schedule from DDPM paper."""
    return torch.linspace(beta_start, beta_end, timesteps)


def cosine_beta_schedule(timesteps, s=0.008):
    """
    Cosine schedule from Improved DDPM paper.
    Better for high-resolution images and smoother transitions.
    """
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0.0001, 0.9999)


def get_beta_schedule(schedule_name, timesteps, beta_start=0.0001, beta_end=0.02):
    """Get beta schedule based on name."""
    if schedule_name == "linear":
        return linear_beta_schedule(timesteps, beta_start, beta_end)
    elif schedule_name == "cosine":
        return cosine_beta_schedule(timesteps)
    else:
        raise ValueError(f"Unknown beta schedule: {schedule_name}")


# 6.2 Precompute diffusion parameters
def prepare_noise_schedule(config):
    """
    Precompute all noise schedule parameters for efficient training.

    These parameters are used in:
    - Forward diffusion q(x_t | x_0): Add noise to clean images
    - Reverse diffusion p(x_{t-1} | x_t): Denoise step by step
    """
    timesteps = config["timesteps"]

    # Get beta schedule
    betas = get_beta_schedule(
        config["beta_schedule"],
        timesteps,
        config["beta_start"],
        config["beta_end"]
    )

    # Compute alpha values
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)

    # Precompute square roots for forward process
    sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
    sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

    # For reverse process
    sqrt_recip_alphas = torch.sqrt(1.0 / alphas)

    # Posterior variance for q(x_{t-1} | x_t, x_0)
    posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

    noise_schedule = {
        "betas": betas,
        "alphas": alphas,
        "alphas_cumprod": alphas_cumprod,
        "alphas_cumprod_prev": alphas_cumprod_prev,
        "sqrt_alphas_cumprod": sqrt_alphas_cumprod,
        "sqrt_one_minus_alphas_cumprod": sqrt_one_minus_alphas_cumprod,
        "sqrt_recip_alphas": sqrt_recip_alphas,
        "posterior_variance": posterior_variance,
        "timesteps": timesteps,
    }

    return noise_schedule


# 6.3 Forward Diffusion Process
def extract(a, t, x_shape):
    """
    Extract coefficients at specified timesteps and reshape for broadcasting.

    Args:
        a: Parameter tensor (e.g., sqrt_alphas_cumprod)
        t: Timestep indices [B]
        x_shape: Shape of x for broadcasting

    Returns:
        Extracted values reshaped to [B, 1, 1, 1, ...]
    """
    batch_size = t.shape[0]
    out = a.gather(-1, t)
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1)))


def q_sample(x_start, t, noise, noise_schedule):
    """
    Forward diffusion process: q(x_t | x_0)

    Add noise to clean images according to the noise schedule.
    Formula: x_t = sqrt(alpha_cumprod_t) * x_0 + sqrt(1 - alpha_cumprod_t) * noise

    Args:
        x_start: Clean images [B, C, H, W]
        t: Timesteps [B]
        noise: Gaussian noise [B, C, H, W]
        noise_schedule: Precomputed schedule parameters

    Returns:
        x_t: Noisy images at timestep t [B, C, H, W]
    """
    sqrt_alphas_cumprod_t = extract(
        noise_schedule["sqrt_alphas_cumprod"], t, x_start.shape
    )
    sqrt_one_minus_alphas_cumprod_t = extract(
        noise_schedule["sqrt_one_minus_alphas_cumprod"], t, x_start.shape
    )

    return sqrt_alphas_cumprod_t * x_start + sqrt_one_minus_alphas_cumprod_t * noise


# 6.4 Create noise schedule
noise_schedule = prepare_noise_schedule(CONFIG)

# Move all tensors to device
for key in noise_schedule:
    if isinstance(noise_schedule[key], torch.Tensor):
        noise_schedule[key] = noise_schedule[key].to(device)

print(f"Noise schedule prepared:")
print(f"  Schedule type: {CONFIG['beta_schedule']}")
print(f"  Timesteps:     {CONFIG['timesteps']}")
print(f"  Beta range:    [{CONFIG['beta_start']}, {CONFIG['beta_end']}]")


# 6.5 Test forward diffusion (optional)
print("\nTesting forward diffusion...")
with torch.no_grad():
    # Get a sample from dataloader
    test_sample = next(iter(train_loader))
    test_tiles = test_sample['tile'][:4].to(device)  # Use 4 samples (tile targets)

    # Test at different timesteps
    test_timesteps = [0, 250, 500, 750, 999]

    for t_val in test_timesteps:
        t = torch.tensor([t_val] * test_tiles.shape[0], device=device)
        noise = torch.randn_like(test_tiles)
        noisy = q_sample(test_tiles, t, noise, noise_schedule)

        # Check value range
        print(f"  t={t_val:4d}: noisy range [{noisy.min():.3f}, {noisy.max():.3f}]")

print("\nSection 6 complete")

## Section 7. UNet Model

In [ ]:
# 7.1 Building Blocks
class SinusoidalPositionEmbeddings(nn.Module):
    """Sinusoidal time embeddings for timestep t."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings


class CrossAttentionBlock(nn.Module):
    """
    Cross-attention block for image conditioning.
    Q from actions features, K/V from character features.

    This allows the UNet to attend to different spatial locations
    in the character encoding when generating actions.
    """
    def __init__(self, channels, context_dim, num_heads=8):
        super().__init__()
        self.channels = channels
        self.context_dim = context_dim
        self.num_heads = num_heads
        self.head_dim = channels // num_heads

        assert channels % num_heads == 0, "channels must be divisible by num_heads"

        self.norm = nn.GroupNorm(32, channels)
        self.context_norm = nn.GroupNorm(32, context_dim)

        # Q from actions features, K/V from character features
        self.to_q = nn.Conv2d(channels, channels, kernel_size=1)
        self.to_k = nn.Conv2d(context_dim, channels, kernel_size=1)
        self.to_v = nn.Conv2d(context_dim, channels, kernel_size=1)
        self.to_out = nn.Conv2d(channels, channels, kernel_size=1)

    def forward(self, x, context):
        """
        Args:
            x: Actions features (B, C, H, W)
            context: Character features (B, context_dim, h, w) - spatial!
        Returns:
            out: (B, C, H, W)
        """
        B, C, H, W = x.shape
        residual = x

        # Normalize
        x = self.norm(x)
        context = self.context_norm(context)

        # Q from actions
        q = self.to_q(x)  # (B, C, H, W)
        q = q.reshape(B, self.num_heads, self.head_dim, H * W)
        q = q.permute(0, 1, 3, 2)  # (B, num_heads, H*W, head_dim)

        # K, V from character features (spatial)
        k = self.to_k(context)  # (B, C, h, w)
        v = self.to_v(context)  # (B, C, h, w)

        _, _, h, w = k.shape
        k = k.reshape(B, self.num_heads, self.head_dim, h * w)
        v = v.reshape(B, self.num_heads, self.head_dim, h * w)
        k = k.permute(0, 1, 3, 2)  # (B, num_heads, h*w, head_dim)
        v = v.permute(0, 1, 3, 2)  # (B, num_heads, h*w, head_dim)

        # Cross-attention: Q from actions, K/V from character
        scale = self.head_dim ** -0.5
        attn = torch.matmul(q, k.transpose(-2, -1)) * scale  # (B, num_heads, H*W, h*w)
        attn = F.softmax(attn, dim=-1)

        # Apply attention
        out = torch.matmul(attn, v)  # (B, num_heads, H*W, head_dim)
        out = out.permute(0, 1, 3, 2).reshape(B, C, H, W)

        # Project and residual
        out = self.to_out(out)
        return out + residual


class ResidualBlock(nn.Module):
    """
    Residual block with time embedding and optional cross-attention.
    """
    def __init__(self, in_channels, out_channels, time_emb_dim,
                 dropout=0.0, use_cross_attn=False, context_dim=None):
        super().__init__()

        self.use_cross_attn = use_cross_attn

        self.norm1 = nn.GroupNorm(32, in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)

        self.time_emb_proj = nn.Linear(time_emb_dim, out_channels)

        self.norm2 = nn.GroupNorm(32, out_channels)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

        if in_channels != out_channels:
            self.residual_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        else:
            self.residual_conv = nn.Identity()

        # Cross-attention for character conditioning
        if use_cross_attn:
            assert context_dim is not None, "context_dim must be provided for cross-attention"
            self.cross_attn = CrossAttentionBlock(out_channels, context_dim)
        else:
            self.cross_attn = None

        self.act = nn.SiLU()

    def forward(self, x, time_emb, context=None):
        """
        Args:
            x: Actions features (B, C, H, W)
            time_emb: Time embeddings (B, time_emb_dim)
            context: Character features (B, context_dim, h, w) or None
        Returns:
            out: (B, C, H, W)
        """
        residual = self.residual_conv(x)

        h = self.norm1(x)
        h = self.act(h)
        h = self.conv1(h)

        # Add time embedding
        time_emb = self.act(time_emb)
        time_emb = self.time_emb_proj(time_emb)
        h = h + time_emb[:, :, None, None]

        h = self.norm2(h)
        h = self.act(h)
        h = self.dropout(h)
        h = self.conv2(h)

        # Cross-attention with character features
        if self.cross_attn is not None and context is not None:
            h = self.cross_attn(h, context)

        return h + residual


class AttentionBlock(nn.Module):
    """Self-attention block for spatial features."""
    def __init__(self, channels, num_heads=8):
        super().__init__()
        self.channels = channels
        self.num_heads = num_heads
        self.head_dim = channels // num_heads

        assert channels % num_heads == 0

        self.norm = nn.GroupNorm(32, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, kernel_size=1)
        self.proj = nn.Conv2d(channels, channels, kernel_size=1)

    def forward(self, x):
        B, C, H, W = x.shape
        residual = x

        x = self.norm(x)

        qkv = self.qkv(x)
        qkv = qkv.reshape(B, 3, self.num_heads, self.head_dim, H * W)
        qkv = qkv.permute(1, 0, 2, 4, 3)
        q, k, v = qkv[0], qkv[1], qkv[2]

        scale = self.head_dim ** -0.5
        attn = torch.matmul(q, k.transpose(-2, -1)) * scale
        attn = F.softmax(attn, dim=-1)

        out = torch.matmul(attn, v)
        out = out.permute(0, 1, 3, 2).reshape(B, C, H, W)

        out = self.proj(out)
        return out + residual


class Downsample(nn.Module):
    """Spatial downsampling by factor of 2."""
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, stride=2, padding=1)

    def forward(self, x):
        return self.conv(x)


class Upsample(nn.Module):
    """Spatial upsampling by factor of 2."""
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode='nearest')
        return self.conv(x)


# 7.2 Main UNet Model (Image-Conditional)
class UNet(nn.Module):
    """
    UNet for image-conditional diffusion models.
    Generates ONE action tile conditioned on 4-view character images.

    Output size is fixed to tile_size x tile_size (64x64).
    """
    def __init__(self, config):
        super().__init__()

        self.config = config
        in_channels = config["in_channels"]
        out_channels = config.get("out_channels", in_channels)
        model_channels = config["unet_channels"]
        channel_mult = config["channel_mult"]
        num_res_blocks = config["num_res_blocks"]
        attention_resolutions = config["attention_resolutions"]
        dropout = config["dropout"]
        num_heads = config["num_heads"]

        # Image conditioning (character features)
        use_cross_attn = config.get("use_cross_attention", True)
        context_dim = config.get("char_feature_dim", 512)

        time_emb_dim = model_channels * 4

        # Time embedding
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(model_channels),
            nn.Linear(model_channels, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )

        # Frame embedding (action/dir/t) injected into time embedding
        # num_frames = 108, plus one "null" id for CFG
        num_frames = config.get("num_frames", 108)
        self.frame_emb = nn.Embedding(num_frames + 1, time_emb_dim)

        # Initial conv
        self.conv_in = nn.Conv2d(in_channels, model_channels, kernel_size=3, padding=1)

        # Fixed tile resolution (square)
        tile_size = config["tile_size"]
        current_h, current_w = tile_size, tile_size

        # Encoder
        self.encoder_blocks = nn.ModuleList()
        current_channels = model_channels
        self.encoder_out_channels = [model_channels]

        for level, mult in enumerate(channel_mult):
            out_channels_level = model_channels * mult

            for i in range(num_res_blocks):
                block_layers = nn.ModuleList()

                # ResBlock with cross-attention
                block_layers.append(
                    ResidualBlock(
                        current_channels,
                        out_channels_level,
                        time_emb_dim,
                        dropout,
                        use_cross_attn=use_cross_attn,
                        context_dim=context_dim
                    )
                )
                current_channels = out_channels_level

                # Self-attention (based on average resolution)
                avg_resolution = (current_h + current_w) // 2
                if avg_resolution in attention_resolutions:
                    block_layers.append(AttentionBlock(current_channels, num_heads))

                self.encoder_blocks.append(block_layers)
                self.encoder_out_channels.append(current_channels)

            # Downsample
            if level != len(channel_mult) - 1:
                self.encoder_blocks.append(nn.ModuleList([Downsample(current_channels)]))
                current_h //= 2
                current_w //= 2
                self.encoder_out_channels.append(current_channels)

        # Bottleneck
        self.mid_block1 = ResidualBlock(
            current_channels,
            current_channels,
            time_emb_dim,
            dropout,
            use_cross_attn=use_cross_attn,
            context_dim=context_dim
        )
        self.mid_attn = AttentionBlock(current_channels, num_heads)
        self.mid_block2 = ResidualBlock(
            current_channels,
            current_channels,
            time_emb_dim,
            dropout,
            use_cross_attn=use_cross_attn,
            context_dim=context_dim
        )

        # Decoder
        self.decoder_blocks = nn.ModuleList()

        for level, mult in reversed(list(enumerate(channel_mult))):
            out_channels_level = model_channels * mult

            for i in range(num_res_blocks + 1):
                skip_channels = self.encoder_out_channels.pop()
                block_layers = nn.ModuleList()

                # ResBlock with cross-attention
                block_layers.append(
                    ResidualBlock(
                        current_channels + skip_channels,
                        out_channels_level,
                        time_emb_dim,
                        dropout,
                        use_cross_attn=use_cross_attn,
                        context_dim=context_dim
                    )
                )
                current_channels = out_channels_level

                # Self-attention
                avg_resolution = (current_h + current_w) // 2
                if avg_resolution in attention_resolutions:
                    block_layers.append(AttentionBlock(current_channels, num_heads))

                self.decoder_blocks.append(block_layers)

                # Upsample
                if level != 0 and i == num_res_blocks:
                    self.decoder_blocks.append(nn.ModuleList([Upsample(current_channels)]))
                    current_h *= 2
                    current_w *= 2

        # Output
        self.conv_out = nn.Sequential(
            nn.GroupNorm(32, current_channels),
            nn.SiLU(),
            nn.Conv2d(current_channels, out_channels, kernel_size=3, padding=1),
        )

    def forward(self, x, t, context=None, cond_ids=None):
        """
        Args:
            x: Noisy tile (B, 4, 64, 64)
            t: Timesteps (B,)
            context: Character features (B, char_feature_dim, 16, 16) - spatial features
            cond_ids: Optional conditioning ids (frame/action/dir/t).
        Returns:
            Predicted noise (B, 4, 64, 64)
        """
        # Time embedding
        time_emb = self.time_mlp(t)

        # Add frame embedding (if provided)
        if cond_ids is not None:
            time_emb = time_emb + self.frame_emb(cond_ids)

        # Initial conv
        h = self.conv_in(x)

        # Encoder
        encoder_features = [h]
        for block_layers in self.encoder_blocks:
            for layer in block_layers:
                if isinstance(layer, ResidualBlock):
                    h = layer(h, time_emb, context)  # Pass character features
                elif isinstance(layer, AttentionBlock):
                    h = layer(h)
                elif isinstance(layer, Downsample):
                    h = layer(h)
            encoder_features.append(h)

        # Bottleneck
        h = self.mid_block1(h, time_emb, context)
        h = self.mid_attn(h)
        h = self.mid_block2(h, time_emb, context)

        # Decoder
        for block_layers in self.decoder_blocks:
            if any(isinstance(layer, ResidualBlock) for layer in block_layers):
                skip = encoder_features.pop()
                h = torch.cat([h, skip], dim=1)

            for layer in block_layers:
                if isinstance(layer, ResidualBlock):
                    h = layer(h, time_emb, context)
                elif isinstance(layer, AttentionBlock):
                    h = layer(h)
                elif isinstance(layer, Upsample):
                    h = layer(h)

        # Output
        return self.conv_out(h)


# 7.3 Weight Initialization
def init_weights(m):
    """Proper weight initialization for stable training."""
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.GroupNorm):
        if m.weight is not None:
            nn.init.ones_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)


# 7.4 Create Model
unet = UNet(CONFIG)
unet.apply(init_weights)
unet = unet.to(device)

total_params = sum(p.numel() for p in unet.parameters())
trainable_params = sum(p.numel() for p in unet.parameters() if p.requires_grad)

print(f"UNet created: {total_params / 1e6:.2f}M parameters")
print(f"  Trainable: {trainable_params / 1e6:.2f}M")
print(f"  Cross-attention: {'Enabled' if CONFIG.get('use_cross_attention') else 'Disabled'}")
print(f"  Context dim: {CONFIG.get('char_feature_dim', 512)} (character features)")
print(f"  Num frames: {CONFIG.get('num_frames', 108)} (+1 null id for CFG)")

# Test forward pass
print("\nTesting UNet forward pass...")
with torch.no_grad():
    test_batch = next(iter(train_loader))
    test_tile = test_batch['tile'][:2].to(device)
    test_fourview = test_batch['fourview'][:2].to(device)

    test_frame_id = test_batch['frame_id'][:2].to(device)

    # Encode character
    test_char_feats = char_encoder(test_fourview)

    # Random timestep
    test_t = torch.randint(0, CONFIG['timesteps'], (2,), device=device)

    # UNet forward
    test_output = unet(test_tile, test_t, context=test_char_feats, cond_ids=test_frame_id)

    print(f"  Input (noisy tile):   {test_tile.shape}")
    print(f"  Character features:   {test_char_feats.shape}")
    print(f"  Frame ids:            {test_frame_id.shape} (min={test_frame_id.min().item()}, max={test_frame_id.max().item()})")
    print(f"  Output (pred noise):  {test_output.shape}")
    print(f"    Forward pass successful!")

print("\nSection 7 complete")

## Section 8. Training Setup

In [ ]:
# 8.1 Trainable Models
# Train both UNet and Character Encoder
# (Unlike text-conditional where CLIP was frozen)

trainable_models = {
    'unet': unet,
    'char_encoder': char_encoder
}

# Count total trainable parameters
total_trainable_params = sum(
    sum(p.numel() for p in model.parameters() if p.requires_grad)
    for model in trainable_models.values()
)

print("Trainable models:")
for name, model in trainable_models.items():
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  {name}: {params / 1e6:.2f}M parameters")
print(f"  Total: {total_trainable_params / 1e6:.2f}M parameters")


# 8.2 Optimizer
# Optimize both UNet and Character Encoder
optimizer = torch.optim.AdamW(
    [
        {'params': unet.parameters()},
        {'params': char_encoder.parameters()},
    ],
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    betas=(0.9, 0.999)
)
print(f"\nOptimizer: AdamW (lr={CONFIG['learning_rate']})")


# 8.3 Learning Rate Scheduler
total_steps = len(train_loader) * CONFIG["num_epochs"]
warmup_steps = min(CONFIG["warmup_steps"], total_steps // 2)

if CONFIG["lr_scheduler"] == "cosine" and total_steps > warmup_steps:
    # Warmup scheduler
    warmup_scheduler = LinearLR(
        optimizer,
        start_factor=0.01,
        end_factor=1.0,
        total_iters=warmup_steps
    )

    # Cosine scheduler
    cosine_steps = total_steps - warmup_steps
    cosine_scheduler = CosineAnnealingLR(
        optimizer,
        T_max=max(1, cosine_steps),
        eta_min=CONFIG["learning_rate"] * 0.1
    )

    # Combine warmup + cosine
    scheduler = SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[warmup_steps]
    )

    print(f"Scheduler: Warmup + CosineAnnealing")
    print(f"  Warmup steps: {warmup_steps}")
    print(f"  Total steps:  {total_steps}")
    print(f"  Cosine steps: {cosine_steps}")
else:
    scheduler = None
    print(f"Scheduler: None (constant learning rate)")


# 8.4 Mixed Precision Training
use_amp = bool(CONFIG.get("use_mixed_precision", False))
scaler = GradScaler(enabled=use_amp)

if use_amp:
    print(f"\nMixed precision: Enabled (FP16)")
else:
    print(f"\nMixed precision: Disabled (FP32)")


# 8.5 Create Directories
checkpoint_dir = Path(CONFIG["checkpoint_dir"])
checkpoint_dir.mkdir(parents=True, exist_ok=True)

log_dir = Path(CONFIG["log_dir"])
log_dir.mkdir(parents=True, exist_ok=True)

sample_dir = Path(CONFIG["sample_dir"])
sample_dir.mkdir(parents=True, exist_ok=True)

print(f"\nDirectories created:")
print(f"  Checkpoints: {checkpoint_dir}")
print(f"  Logs:        {log_dir}")
print(f"  Samples:     {sample_dir}")


# 8.6 Training State (for resuming)
training_state = {
    'epoch': 0,
    'global_step': 0,
    'best_loss': float('inf'),
}

print(f"\nTraining state initialized:")
print(f"  Starting epoch: {training_state['epoch']}")
print(f"  Global step:    {training_state['global_step']}")


# 8.7 Resume from Checkpoint (if enabled)
if CONFIG.get("resume_training", False):
    # Find latest checkpoint
    checkpoints = sorted(checkpoint_dir.glob("checkpoint_epoch_*.pt"))

    if checkpoints:
        latest_checkpoint = checkpoints[-1]
        print(f"\nResuming from checkpoint: {latest_checkpoint}")

        checkpoint = torch.load(latest_checkpoint, map_location=device)

        # Load model states
        unet.load_state_dict(checkpoint['unet_state_dict'])
        char_encoder.load_state_dict(checkpoint['char_encoder_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

        if scheduler is not None and 'scheduler_state_dict' in checkpoint:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

        if scaler is not None and 'scaler_state_dict' in checkpoint:
            scaler.load_state_dict(checkpoint['scaler_state_dict'])

        # Load training state
        training_state['epoch'] = checkpoint.get('epoch', 0) + 1
        training_state['global_step'] = checkpoint.get('global_step', 0)
        training_state['best_loss'] = checkpoint.get('best_loss', float('inf'))

        print(f"  Resumed from epoch {training_state['epoch'] - 1}")
        print(f"  Global step: {training_state['global_step']}")
        print(f"  Best loss: {training_state['best_loss']:.6f}")
    else:
        print(f"\nNo checkpoint found, starting fresh training")
        CONFIG["resume_training"] = False
else:
    print(f"\nStarting fresh training")


print("\nSection 8 complete")

## Section 9. DDIM Sampling

In [ ]:
# 9.1 DDIM Sampling Functions
@torch.no_grad()
def ddim_sample_step(unet, x, t, t_prev, noise_schedule, context=None, cond_ids=None, eta=0.0):
    """
    Single DDIM sampling step.

    Args:
        unet: UNet model
        x: Current noisy image (B, C, H, W)
        t: Current timestep (B,)
        t_prev: Previous timestep (scalar or tensor)
        noise_schedule: Noise schedule parameters
        context: Character features (B, C, h, w) or None
        cond_ids: Optional conditioning ids (frame/action/dir/t)
        eta: Stochasticity parameter (0.0 = deterministic DDIM)

    Returns:
        x_prev: Less noisy image
    """
    device = x.device
    alphas_cumprod = noise_schedule['alphas_cumprod']

    # Extract alpha values
    t_index = t[0].item()
    alpha_t = alphas_cumprod[t_index]

    if isinstance(t_prev, torch.Tensor):
        if t_prev.item() >= 0:
            alpha_prev = alphas_cumprod[t_prev.item()]
        else:
            alpha_prev = torch.tensor(1.0, device=device)
    else:
        if t_prev >= 0:
            alpha_prev = alphas_cumprod[t_prev]
        else:
            alpha_prev = torch.tensor(1.0, device=device)

    # Reshape for broadcasting
    alpha_t = alpha_t.view(1, 1, 1, 1)
    alpha_prev = alpha_prev.view(1, 1, 1, 1)

    # Predict noise (with character conditioning)
    predicted_noise = unet(x, t, context=context, cond_ids=cond_ids)

    # Predict x0
    x0_pred = (x - torch.sqrt(1 - alpha_t) * predicted_noise) / torch.sqrt(alpha_t)
    x0_pred = torch.clamp(x0_pred, -1.0, 1.0)

    # Compute variance
    sigma_t = eta * torch.sqrt((1 - alpha_prev) / (1 - alpha_t)) * torch.sqrt(1 - alpha_t / alpha_prev)

    # Compute x_{t-1}
    noise = torch.randn_like(x) if eta > 0 else 0
    x_prev = torch.sqrt(alpha_prev) * x0_pred + \
             torch.sqrt(1 - alpha_prev - sigma_t**2) * predicted_noise + \
             sigma_t * noise

    return x_prev


@torch.no_grad()
def ddim_sample_tile(unet, char_encoder, fourview, noise_schedule, device,
                     frame_id, cfg_scale=7.5, ddim_steps=50, eta=0.0, tile_size=64):
    """
    DDIM sampling with Classifier-Free Guidance for image-conditional generation.

    This generates ONE 64x64 tile given:
      - fourview (character)
      - frame_id (which tile to generate)

    Args:
        unet: UNet model
        char_encoder: Character encoder model
        fourview: 4-view character images (B, 4, 128, 128)
        noise_schedule: Noise schedule
        device: Device
        frame_id: Conditioning ids (B,) - integer [0..107]
        cfg_scale: Guidance scale (1.0 = no guidance, higher = stronger conditioning)
        ddim_steps: Number of sampling steps
        eta: Stochasticity (0.0 = deterministic)
        tile_size: Output tile size (64)

    Returns:
        Generated tiles (B, 4, 64, 64)
    """
    batch_size = fourview.shape[0]
    total_timesteps = noise_schedule['timesteps']

    shape = (batch_size, 4, tile_size, tile_size)  # RGBA tile

    # Create timestep sequence
    c = total_timesteps // ddim_steps
    ddim_timesteps = torch.arange(0, total_timesteps, c, device=device)
    ddim_timesteps = torch.cat([ddim_timesteps, torch.tensor([total_timesteps - 1], device=device)])

    # Start from noise
    x = torch.randn(shape, device=device)

    # Encode character features
    char_features = char_encoder(fourview)  # (B, 512, 16, 16)

    # Classifier-Free Guidance setup
    use_cfg = cfg_scale > 1.0

    if use_cfg:
        # Create unconditional context (null image features)
        uncond_char_features = torch.zeros_like(char_features)

    # Reverse process (denoising)
    for i in reversed(range(len(ddim_timesteps))):
        t = ddim_timesteps[i]
        t_prev = ddim_timesteps[i - 1] if i > 0 else torch.tensor(-1, device=device)
        t_batch = t.repeat(batch_size)

        if use_cfg:
            # Predict with character conditioning
            noise_pred_cond = unet(x, t_batch, context=char_features, cond_ids=frame_id)

            # Predict without conditioning (unconditional)
            noise_pred_uncond = unet(x, t_batch, context=uncond_char_features, cond_ids=frame_id)

            # Combine predictions
            noise_pred = noise_pred_uncond + cfg_scale * (noise_pred_cond - noise_pred_uncond)

            # Manual DDIM step with combined prediction
            alpha_t = noise_schedule['alphas_cumprod'][t.item()].view(1, 1, 1, 1)

            if isinstance(t_prev, torch.Tensor):
                if t_prev.item() >= 0:
                    alpha_prev = noise_schedule['alphas_cumprod'][t_prev.item()].view(1, 1, 1, 1)
                else:
                    alpha_prev = torch.tensor(1.0, device=device).view(1, 1, 1, 1)
            else:
                if t_prev >= 0:
                    alpha_prev = noise_schedule['alphas_cumprod'][t_prev].view(1, 1, 1, 1)
                else:
                    alpha_prev = torch.tensor(1.0, device=device).view(1, 1, 1, 1)

            x0_pred = (x - torch.sqrt(1 - alpha_t) * noise_pred) / torch.sqrt(alpha_t)
            x0_pred = torch.clamp(x0_pred, -1.0, 1.0)

            sigma_t = eta * torch.sqrt((1 - alpha_prev) / (1 - alpha_t)) * torch.sqrt(1 - alpha_t / alpha_prev)
            noise = torch.randn_like(x) if eta > 0 else 0
            x = torch.sqrt(alpha_prev) * x0_pred + \
                torch.sqrt(1 - alpha_prev - sigma_t**2) * noise_pred + \
                sigma_t * noise
        else:
            # Standard DDIM (no guidance)
            x = ddim_sample_step(unet, x, t_batch, t_prev, noise_schedule, char_features, cond_ids=frame_id, eta=eta)

    return torch.clamp(x, -1.0, 1.0)

In [ ]:
# 9.2 Assemble full action sheet from generated tiles
@torch.no_grad()
def assemble_action_sheet_from_tiles(tiles_by_frame, tile_size=64, sheet_rows=12, sheet_cols=9):
    """
    Assemble full 576x768 action sheet from tiles.

    Args:
        tiles_by_frame: dict mapping frame_id -> tile tensor (4, 64, 64) in [-1, 1]
        tile_size: 64
        sheet_rows: 12
        sheet_cols: 9

    Returns:
        sheet: tensor (4, 768, 576) in [-1, 1]
    """
    sheet_h = sheet_rows * tile_size
    sheet_w = sheet_cols * tile_size

    sheet = torch.zeros(4, sheet_h, sheet_w, device=next(iter(tiles_by_frame.values())).device)

    for frame_id, tile in tiles_by_frame.items():
        # frame_id = action_id * 36 + dir_id * 9 + t_id
        row = frame_id // sheet_cols
        col = frame_id % sheet_cols

        y0 = row * tile_size
        x0 = col * tile_size
        sheet[:, y0:y0+tile_size, x0:x0+tile_size] = tile

    return sheet


@torch.no_grad()
def generate_full_actions_sheet(unet, char_encoder, fourview, noise_schedule, device,
                               cfg_scale=7.5, ddim_steps=50, eta=0.0, config=None):
    """
    Generate the full 576x768 actions sheet by sampling all valid tiles and assembling.

    Valid tiles:
      - walk:   4x9
      - thrust: 4x8
      - slash:  4x6
    """
    assert config is not None, "config must be provided"

    tile_size = config["tile_size"]
    sheet_cols = config["sheet_cols"]
    sheet_rows = config["sheet_rows"]

    # Precompute valid frame_ids (same rule as dataset)
    actions = config["actions"]
    dirs = config["dirs"]
    action_cols = config["action_cols"]

    action_to_id = {a: i for i, a in enumerate(actions)}

    tiles_by_frame = {}

    # Generate tiles one frame at a time
    for action_name in actions:
        a_id = action_to_id[action_name]
        max_t = action_cols[action_name]
        for d_id in range(len(dirs)):
            for t_id in range(max_t):
                frame_id = torch.full((fourview.shape[0],), a_id * (4 * sheet_cols) + d_id * sheet_cols + t_id,
                                      device=device, dtype=torch.long)
                tile = ddim_sample_tile(
                    unet=unet,
                    char_encoder=char_encoder,
                    fourview=fourview,
                    noise_schedule=noise_schedule,
                    device=device,
                    frame_id=frame_id,
                    cfg_scale=cfg_scale,
                    ddim_steps=ddim_steps,
                    eta=eta,
                    tile_size=tile_size
                )
                # tile: (B, 4, 64, 64) - store per sample later
                tiles_by_frame[(a_id, d_id, t_id)] = tile

    # Assemble per sample in batch
    sheets = []
    for b in range(fourview.shape[0]):
        tiles_map = {}
        for (a_id, d_id, t_id), tile_b in tiles_by_frame.items():
            frame_id_int = a_id * (4 * sheet_cols) + d_id * sheet_cols + t_id
            tiles_map[frame_id_int] = tile_b[b]  # (4, 64, 64)
        sheet = assemble_action_sheet_from_tiles(tiles_map, tile_size=tile_size, sheet_rows=sheet_rows, sheet_cols=sheet_cols)
        sheets.append(sheet)

    sheets = torch.stack(sheets, dim=0)  # (B, 4, 768, 576)
    return torch.clamp(sheets, -1.0, 1.0)

In [ ]:
# 9.3 Quick Validation Sampling
@torch.no_grad()
def quick_validation_sample(unet, char_encoder, train_loader, noise_schedule, device, epoch, save_dir):
    """
    Generate samples for quick validation during training.
    Uses fixed 4-view images from training set.

    Validation strategy (important):
      1) Tile-level grid: verify frame_id conditioning actually changes outputs.
      2) (Optional) Full sheet: assemble 576x768 for a coarse visual check (better after training longer).

    Args:
        unet: UNet model
        char_encoder: Character encoder
        train_loader: DataLoader to get 4-view samples
        noise_schedule: Noise schedule
        device: Device
        epoch: Current epoch
        save_dir: Save directory
    """
    print(f"Validation sampling at epoch {epoch}...")

    unet.eval()
    char_encoder.eval()

    epoch_dir = Path(save_dir) / f"epoch_{epoch}"
    epoch_dir.mkdir(parents=True, exist_ok=True)

    # ---------------------------
    # Part A: Tile-level validation
    # ---------------------------
    print("  [A] Tile-level validation (frame_id conditioning)")

    test_batch = next(iter(train_loader))
    # Use ONE character for clear comparison (same appearance, different frame_id)
    test_fourview = test_batch['fourview'][:1].to(device)   # (1, 4, 128, 128)
    test_filename = test_batch['filename'][:1][0]

    # Choose a set of frame_ids to visualize (e.g., first row 0..8)
    sheet_cols = CONFIG["sheet_cols"]      # 9
    num_show = sheet_cols                  # show 9 tiles in one row
    frame_ids = list(range(num_show))      # 0..8 (walk, one dir, t=0..8 in the mapping)

    tiles = []
    for fid in frame_ids:
        frame_id = torch.tensor([fid], device=device, dtype=torch.long)  # (1,)
        tile = ddim_sample_tile(
            unet=unet,
            char_encoder=char_encoder,
            fourview=test_fourview,
            noise_schedule=noise_schedule,
            device=device,
            frame_id=frame_id,
            cfg_scale=CONFIG.get("cfg_scale", 1.0),   # keep small during training validation
            ddim_steps=CONFIG.get("default_ddim_steps", 50),
            eta=0.0,
            tile_size=CONFIG["tile_size"]
        )
        tiles.append(tile[0])  # (4, 64, 64)

    tiles = torch.stack(tiles, dim=0)  # (N, 4, 64, 64)

    # Save ONLY one grid (avoid duplicate per-tile images and extra grids)
    tiles_vis = (tiles + 1.0) / 2.0
    tiles_vis = torch.clamp(tiles_vis, 0.0, 1.0)
    grid_img = make_grid(tiles_vis, nrow=num_show, padding=2, pad_value=1.0)

    grid_path = epoch_dir / f"epoch{epoch}_tile_grid_{Path(test_filename).stem}.png"
    save_image(grid_img, grid_path)

    print(f"  Saved tile grid: {grid_path.name}")

    # ---------------------------
    # Part B: Full sheet validation (optional)
    # ---------------------------
    save_full_sheet = CONFIG.get("save_full_sheet", False)
    every_n = max(1, int(CONFIG.get("full_sheet_every_epochs", 5)))

    do_full_sheet = save_full_sheet and (epoch % every_n == 0)

    if do_full_sheet:
        print("  [B] Full-sheet validation (assemble 576x768)")

        test_fourview_4 = test_batch['fourview'][:4].to(device)
        test_filenames_4 = test_batch['filename'][:4]

        generated_actions = generate_full_actions_sheet(
            unet=unet,
            char_encoder=char_encoder,
            fourview=test_fourview_4,
            noise_schedule=noise_schedule,
            device=device,
            cfg_scale=CONFIG.get("cfg_scale", 7.5),
            ddim_steps=CONFIG.get("default_ddim_steps", 50),
            eta=0.0,
            config=CONFIG
        )

        save_generated_images_with_fourview(
            fourview=test_fourview_4,
            generated_actions=generated_actions,
            filenames=test_filenames_4,
            save_dir=epoch_dir,
            filename_prefix=f"epoch{epoch}"
        )

        print(f"  Saved full sheets: {epoch_dir}")

    unet.train()
    char_encoder.train()

    return True

In [ ]:
# 9.4 Save Generated Images
def save_generated_images(images, save_dir, filename_prefix="sample"):
    """Save generated images to disk."""
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    # Convert from [-1, 1] to [0, 1]
    images = (images + 1.0) / 2.0
    images = torch.clamp(images, 0.0, 1.0)

    # Save individual images
    for i, img in enumerate(images):
        filepath = save_dir / f"{filename_prefix}_{i:04d}.png"
        save_image(img, filepath)

    # Save grid
    if len(images) > 1:
        grid_img = make_grid(images, nrow=2, padding=2, pad_value=1.0)
        grid_path = save_dir / f"{filename_prefix}_grid.png"
        save_image(grid_img, grid_path)

    print(f"  Saved {len(images)} images")


def save_generated_images_with_fourview(fourview, generated_actions, filenames, save_dir, filename_prefix="sample"):
    """
    Save generated actions images alongside their 4-view inputs.

    Args:
        fourview: Input 4-view images (B, 4, 128, 128)
        generated_actions: Generated actions (B, 4, 768, 576)
        filenames: Original filenames
        save_dir: Save directory
        filename_prefix: Filename prefix
    """
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    # Convert from [-1, 1] to [0, 1]
    fourview = (fourview + 1.0) / 2.0
    generated_actions = (generated_actions + 1.0) / 2.0
    fourview = torch.clamp(fourview, 0.0, 1.0)
    generated_actions = torch.clamp(generated_actions, 0.0, 1.0)

    # Save individual pairs
    for i, (fv, actions, fname) in enumerate(zip(fourview, generated_actions, filenames)):
        # Save 4-view input
        fv_path = save_dir / f"{filename_prefix}_{i:04d}_input_{fname}"
        save_image(fv, fv_path)

        # Save generated actions
        actions_path = save_dir / f"{filename_prefix}_{i:04d}_output_{fname}"
        save_image(actions, actions_path)

    # Save comparison grid (4-view on left, actions on right)
    if len(fourview) > 1:
        # Resize 4-view to match actions height for better comparison
        fv_resized = F.interpolate(fourview, size=generated_actions.shape[-2:], mode='nearest')

        # Interleave: [4view, actions, 4view, actions, ...]
        comparison = []
        for fv, actions in zip(fv_resized, generated_actions):
            comparison.append(fv)
            comparison.append(actions)

        comparison = torch.stack(comparison)
        grid_img = make_grid(comparison, nrow=2, padding=2, pad_value=1.0)
        grid_path = save_dir / f"{filename_prefix}_comparison_grid.png"
        save_image(grid_img, grid_path)

    # Save filenames reference
    filenames_file = save_dir / f"{filename_prefix}_filenames.txt"
    with open(filenames_file, 'w', encoding='utf-8') as f:
        for i, fname in enumerate(filenames):
            f.write(f"{i}: {fname}\n")

    print(f"  Saved {len(fourview)} sample pairs")


print("\nSection 9 complete")

## Section 10. Training Loop

In [ ]:
# 10.1 Training Loop with Image Conditioning
def train_one_epoch(epoch, unet, char_encoder, train_loader,
                    optimizer, scheduler, scaler, noise_schedule, device, config):
    """
    Train for one epoch with image conditioning.
    Implements Classifier-Free Guidance training.

    Trains both UNet and Character Encoder end-to-end.
    """
    unet.train()
    char_encoder.train()  # Both models in training mode

    epoch_loss = 0.0
    num_batches = 0

    # Progress bar
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['num_epochs']}")

    for batch_idx, batch in enumerate(pbar):
        start_time = time.time()

        # Get paired images
        fourview = batch['fourview'].to(device)      # (B, 4, 128, 128)
        tile = batch['tile'].to(device)              # (B, 4, 64, 64)
        frame_id = batch['frame_id'].to(device)      # (B,)
        batch_size = fourview.shape[0]

        # Encode character features
        char_features = char_encoder(fourview)  # (B, 512, 16, 16)

        # Classifier-Free Guidance: randomly drop conditioning
        cfg_drop_prob = config.get("cfg_drop_prob", 0.1)
        drop_mask = torch.rand(batch_size, device=device) < cfg_drop_prob

        if drop_mask.any():
            # Zero out character features for dropped samples
            char_features = char_features.clone()
            char_features[drop_mask] = 0

            # Also replace frame_id with a special "null" id
            # (so UNet learns unconditional generation as well)
            frame_id = frame_id.clone()
            frame_id[drop_mask] = config["num_frames"]  # null id

        # Random timesteps
        t = torch.randint(0, config["timesteps"], (batch_size,), device=device).long()

        # Add noise to tile
        noise = torch.randn_like(tile)
        noisy_tile = q_sample(tile, t, noise, noise_schedule)

        # Predict noise (with conditioning)
        optimizer.zero_grad(set_to_none=True)

        if config["use_mixed_precision"]:
            with autocast():
                noise_pred = unet(noisy_tile, t, context=char_features, cond_ids=frame_id)
                loss = F.mse_loss(noise_pred, noise)
            scaler.scale(loss).backward()

            if config["gradient_clip"] > 0:
                scaler.unscale_(optimizer)
                # Clip gradients for both models
                torch.nn.utils.clip_grad_norm_(unet.parameters(), config["gradient_clip"])
                torch.nn.utils.clip_grad_norm_(char_encoder.parameters(), config["gradient_clip"])

            scaler.step(optimizer)
            scaler.update()
        else:
            noise_pred = unet(noisy_tile, t, context=char_features, cond_ids=frame_id)
            loss = F.mse_loss(noise_pred, noise)

            loss.backward()

            if config["gradient_clip"] > 0:
                # Clip gradients for both models
                torch.nn.utils.clip_grad_norm_(unet.parameters(), config["gradient_clip"])
                torch.nn.utils.clip_grad_norm_(char_encoder.parameters(), config["gradient_clip"])

            optimizer.step()

        # Update learning rate
        if scheduler is not None:
            scheduler.step()

        # Track loss
        epoch_loss += loss.item()
        num_batches += 1

        # Update progress bar
        current_lr = optimizer.param_groups[0]['lr']
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'avg': f'{epoch_loss/num_batches:.4f}',
            'lr': f'{current_lr:.6f}'
        })

        # Periodic logging
        if batch_idx % config["log_every"] == 0:
            batch_time = time.time() - start_time
            print(f"\n[Epoch {epoch+1}] [Batch {batch_idx}/{len(train_loader)}] "
                  f"Loss: {loss.item():.4f}, Avg: {epoch_loss/num_batches:.4f}, "
                  f"LR: {current_lr:.6f}, Time: {batch_time:.2f}s")

    return epoch_loss / num_batches

In [ ]:
# 10.2 Checkpoint Management
def save_checkpoint(epoch, unet, char_encoder, optimizer, scheduler, scaler, loss,
                   checkpoint_dir, loss_history=None, best_loss=None,
                   global_step=0, keep_last_n=3):
    """Save training checkpoint with both models."""
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(exist_ok=True, parents=True)

    checkpoint = {
        'epoch': epoch,
        'global_step': global_step,
        'unet_state_dict': unet.state_dict(),
        'char_encoder_state_dict': char_encoder.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'scaler_state_dict': scaler.state_dict() if scaler is not None else None,
        'loss': loss,
        'best_loss': best_loss,
        'loss_history': loss_history,
        'config': CONFIG,
    }

    # Save epoch checkpoint
    checkpoint_path = checkpoint_dir / f"checkpoint_epoch_{epoch+1}.pt"
    torch.save(checkpoint, checkpoint_path)
    print(f"  Saved checkpoint: {checkpoint_path.name}")

    # Update latest
    latest_path = checkpoint_dir / "checkpoint_latest.pt"
    torch.save(checkpoint, latest_path)

    # Check if best
    is_best = False
    if best_loss is None or loss < best_loss:
        best_path = checkpoint_dir / "checkpoint_best.pt"
        torch.save(checkpoint, best_path)

        # Save model-only versions (for inference)
        model_only_dir = checkpoint_dir.parent / "final_models"
        model_only_dir.mkdir(exist_ok=True, parents=True)

        torch.save(unet.state_dict(), model_only_dir / "unet_best.pt")
        torch.save(char_encoder.state_dict(), model_only_dir / "char_encoder_best.pt")

        prev_loss_str = f"{best_loss:.4f}" if best_loss is not None else "N/A"
        print(f"  New best model: {loss:.4f} (prev: {prev_loss_str})")

        is_best = True
        best_loss = loss

    # Cleanup old checkpoints
    all_checkpoints = sorted(checkpoint_dir.glob("checkpoint_epoch_*.pt"))
    if len(all_checkpoints) > keep_last_n:
        for old_checkpoint in all_checkpoints[:-keep_last_n]:
            print(f"  Removing old checkpoint: {old_checkpoint.name}")
            old_checkpoint.unlink()

    return best_loss


def load_checkpoint(checkpoint_path, unet, char_encoder, optimizer=None,
                   scheduler=None, scaler=None, device='cuda'):
    """Load checkpoint and restore training state."""
    print(f"Loading checkpoint: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    # Load model states
    unet.load_state_dict(checkpoint['unet_state_dict'])
    char_encoder.load_state_dict(checkpoint['char_encoder_state_dict'])

    # Load optimizer
    if optimizer is not None and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    # Load scheduler
    if scheduler is not None and checkpoint.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    # Load scaler
    if scaler is not None and checkpoint.get('scaler_state_dict') is not None:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])

    # Load training state
    epoch = checkpoint['epoch']
    global_step = checkpoint.get('global_step', 0)
    loss = checkpoint['loss']
    best_loss = checkpoint.get('best_loss', loss)
    loss_history = checkpoint.get('loss_history', [])

    print(f"  Checkpoint loaded: {Path(checkpoint_path).name}")
    print(f"    Epoch: {epoch+1}")
    print(f"    Global step: {global_step}")
    print(f"    Loss: {loss:.4f}")
    print(f"    Best loss: {best_loss:.4f}")
    print(f"    History length: {len(loss_history)} epochs")

    return epoch, global_step, loss, loss_history, best_loss


def save_loss_history(loss_history, save_path):
    """Save loss history to CSV."""
    import csv
    from datetime import datetime

    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    with open(save_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['epoch', 'loss', 'timestamp'])

        for epoch, loss in enumerate(loss_history, start=1):
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            writer.writerow([epoch, f'{loss:.6f}', timestamp])

    print(f"  Loss history saved: {save_path.name}")

In [ ]:
# 10.3 Main Training Function
def train(unet, char_encoder, train_loader, optimizer, scheduler, scaler,
          noise_schedule, device, config, training_state):
    """Main training loop with image conditioning."""
    start_epoch = training_state['epoch']
    global_step = training_state['global_step']
    best_loss = training_state['best_loss']

    print(f"\nStarting training:")
    print(f"  Model: Image-Conditional UNet + Character Encoder (Tile Training)")
    print(f"  Target: {config['tile_size']}x{config['tile_size']} tile")
    print(f"  Frames: {config['num_frames']} (+1 null id for CFG)")
    print(f"  UNet params: {sum(p.numel() for p in unet.parameters()) / 1e6:.2f}M")
    print(f"  Char Encoder params: {sum(p.numel() for p in char_encoder.parameters()) / 1e6:.2f}M")
    print(f"  CFG drop prob: {config.get('cfg_drop_prob', 0.1)}")
    print(f"  CFG scale: {config.get('cfg_scale', 7.5)}")
    print(f"  Total epochs: {config['num_epochs']}")
    print(f"  Starting from: epoch {start_epoch+1}, step {global_step}\n")

    loss_history = []

    try:
        for epoch in range(start_epoch, config["num_epochs"]):
            epoch_start = time.time()

            # Re-sample tiles each epoch (IMPORTANT for huge tile datasets)
            train_loader = create_dataloader(config)
            print(f"  Batches/epoch (this epoch): {len(train_loader)}")

            # Train one epoch
            avg_loss = train_one_epoch(
                epoch, unet, char_encoder, train_loader,
                optimizer, scheduler, scaler, noise_schedule, device, config
            )

            loss_history.append(avg_loss)
            global_step += len(train_loader)
            epoch_time = time.time() - epoch_start

            print(f"\nEpoch {epoch+1} summary:")
            print(f"  Loss: {avg_loss:.4f}")
            print(f"  Time: {epoch_time/60:.1f}min")
            print(f"  Global step: {global_step}")

            # Save checkpoint
            if (epoch + 1) % config["save_every_epochs"] == 0:
                best_loss = save_checkpoint(
                    epoch, unet, char_encoder, optimizer, scheduler, scaler, avg_loss,
                    checkpoint_dir, loss_history, best_loss, global_step,
                    keep_last_n=config.get("keep_last_n", 3)
                )

            # Quick validation sampling
            if (epoch + 1) % config.get("sample_every_epochs", 5) == 0:
                quick_validation_sample(
                    unet, char_encoder, train_loader, noise_schedule,
                    device, epoch+1, sample_dir
                )

            # Save loss CSV
            loss_csv_path = log_dir / "loss_history.csv"
            save_loss_history(loss_history, loss_csv_path)

    except KeyboardInterrupt:
        print("\n\nTraining interrupted by user")
        print("Saving checkpoint...")

        best_loss = save_checkpoint(
            epoch, unet, char_encoder, optimizer, scheduler, scaler, avg_loss,
            checkpoint_dir, loss_history, best_loss, global_step,
            keep_last_n=config.get("keep_last_n", 3)
        )

        loss_csv_path = log_dir / "loss_history.csv"
        save_loss_history(loss_history, loss_csv_path)

        print("Checkpoint saved. Training can be resumed.")
        return loss_history

    # Final checkpoint
    print("\nSaving final checkpoint...")
    best_loss = save_checkpoint(
        config["num_epochs"]-1, unet, char_encoder, optimizer, scheduler, scaler,
        loss_history[-1] if loss_history else 0.0,
        checkpoint_dir, loss_history, best_loss, global_step,
        keep_last_n=config.get("keep_last_n", 3)
    )

    loss_csv_path = log_dir / "loss_history.csv"
    save_loss_history(loss_history, loss_csv_path)

    print(f"\n{'='*60}")
    print(f"Training complete!")
    print(f"{'='*60}")
    print(f"  Total epochs: {config['num_epochs']}")
    print(f"  Final loss: {loss_history[-1]:.4f}")
    print(f"  Best loss: {best_loss:.4f}")
    print(f"  Loss CSV: {loss_csv_path}")
    print(f"  Checkpoints: {checkpoint_dir}")
    print(f"  Samples: {sample_dir}")
    print(f"{'='*60}\n")

    return loss_history

In [ ]:
# 10.4 Start Training
START_TRAINING = CONFIG.get("start_training", False)
RESUME_TRAINING = CONFIG.get("resume_training", False)

if START_TRAINING and not RESUME_TRAINING:
    # Fresh training
    print("="*60)
    print("STARTING FRESH TRAINING")
    print("="*60)

    loss_history = train(
        unet, char_encoder, train_loader,
        optimizer, scheduler, scaler, noise_schedule,
        device, CONFIG, training_state
    )

    print("Training completed successfully!")

elif RESUME_TRAINING:
    # Resume training
    print("="*60)
    print("RESUMING TRAINING FROM CHECKPOINT")
    print("="*60)

    checkpoint_path = checkpoint_dir / "checkpoint_latest.pt"

    if checkpoint_path.exists():
        # Load checkpoint
        epoch, global_step, last_loss, previous_loss_history, best_loss = load_checkpoint(
            checkpoint_path, unet, char_encoder, optimizer, scheduler, scaler, device
        )

        # Update training state
        training_state['epoch'] = epoch + 1
        training_state['global_step'] = global_step
        training_state['best_loss'] = best_loss

        print(f"\nResuming from epoch {epoch+1}")

        # Continue training
        new_loss_history = train(
            unet, char_encoder, train_loader,
            optimizer, scheduler, scaler, noise_schedule,
            device, CONFIG, training_state
        )

        # Combine loss history
        combined_loss_history = previous_loss_history + new_loss_history
        combined_csv_path = log_dir / "loss_history_combined.csv"
        save_loss_history(combined_loss_history, combined_csv_path)

        print("Resume training completed!")
    else:
        print(f"Checkpoint not found: {checkpoint_path}")
        print("Starting fresh training instead...")

        loss_history = train(
            unet, char_encoder, train_loader,
            optimizer, scheduler, scaler, noise_schedule,
            device, CONFIG, training_state
        )

else:
    print("="*60)
    print("TRAINING NOT STARTED")
    print("="*60)
    print("\nTo start training, set one of:")
    print("  CONFIG['start_training'] = True   # Fresh training")
    print("  CONFIG['resume_training'] = True  # Resume from checkpoint")
    print("\nCurrent settings:")
    print(f"  start_training:  {START_TRAINING}")
    print(f"  resume_training: {RESUME_TRAINING}")

print("\nSection 10 complete")